In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q scipy
!pip install -q h5py
!pip install -q matplotlib
!pip install -q torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchaudio==0.13.1 --extra-index-url https://download.pytorch.org/whl/cu117
!pip install -q transformers
!pip install -q fuzzy_match
!pip install -q nltk
!pip install -q rouge
!pip install -q diffusers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 13.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 62.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchdata 0.7.0 requires torch==2.1.0, but you have torch 1.13.1+cu117 which is incompatible.
torchtext 0.16.0 requires torch==2.1.0, but you have torch 1.13.1+cu117 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 54.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 30.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 94.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 34.0 MB/s eta 0:0

In [3]:
%cd /content/drive/MyDrive/Research/FINAL/Code/MMMM

/content/drive/MyDrive/Research/FINAL/Code/MMMM


In [4]:
import sys

sys.path.append("./training")
sys.path.append("./testing")
sys.path.append("./utils")
sys.path.append("./ZuCo")

from DSG import *
from load_data import *
from data import *
from dataloader import *

In [ ]:
ZuCo_data = load_txt_data("./data/ZuCo")
master_eeg, master_embeds = ZuCo_data["data"], ZuCo_data["targets"]
del ZuCo_data
Brain2Image_data = load_img_data("./data/Brain2Image")
image_eeg_labels, img_net_dict = Brain2Image_data["data"], Brain2Image_data["targets"]
del Brain2Image_data

In [46]:
zuco_dataloader = ZuCoDataloader(master_eeg["train"], master_embeds["train"], bsz=64, drop_last=True)
zuco_test_dataloader = ZuCoDataloader(master_eeg["test"], master_embeds["test"], bsz=64, drop_last=True)
image_net_dataloader = ImageNetDataloader(image_eeg_labels["train"], img_net_dict, bsz=1, drop_last=True)
image_net_test_dataloader = ImageNetDataloader(image_eeg_labels["test"], img_net_dict, bsz=1, drop_last=True)

In [53]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda"

In [82]:
class TaskHead(nn.Module):
    def __init__(self, input_dim=512, output_dim=1024, hidden_dim=1024, num_layers=2, device="cuda"):
        super(TaskHead, self).__init__()
        self.hidden_layers = nn.ModuleList()
        self.hidden_layers.append(nn.Linear(input_dim, hidden_dim))
        for i in range(num_layers - 1):
            self.hidden_layers.append(nn.Linear(hidden_dim, hidden_dim))
        self.output_layer = nn.Linear(hidden_dim, output_dim)
        self.activation = nn.ReLU()
        self.device = device

    def forward(self, x):
        for layer in self.hidden_layers:
            x = self.activation(layer(x))
        x = self.activation(self.output_layer(x))
        return x

class EEGEncoder(nn.Module):
    def __init__(self, enc_feat=1024, dec_emb_sz=768, enc_nhead=8, enc_dim_ff=2048, num_enc_layers=8, device="cuda"):
        super(EEGEncoder, self).__init__()

        self.device = device

        self.task_heads = {}

        self.encoder_layer = nn.TransformerEncoderLayer(d_model=enc_feat, nhead=enc_nhead, dim_feedforward=enc_dim_ff, batch_first=True)
        self.encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=num_enc_layers)
        self.fc_proj = nn.Linear(enc_feat, dec_emb_sz)

    def add_task(self, task_name, task_head):
        self.task_heads[task_name] = task_head

    def forward(self, mode, input_data_batch=None, input_masks_batch=None, input_masks_invert=None, pool_result=False):
        encoded_embedding = self.task_heads[mode](input_data_batch)
        encoded_embedding = self.encoder(encoded_embedding, src_key_padding_mask = input_masks_invert)
        encoded_embedding = F.relu(self.fc_proj(encoded_embedding))

        if pool_result:
            pooler = torch.zeros(encoded_embedding.shape[0:1]+encoded_embedding.shape[-1:]).to(device)
            for i in range(encoded_embedding.shape[0]):
              for j in range(encoded_embedding.shape[-2]):
                  pooler[i] += encoded_embedding[i][j][:]
              pooler[i] /= encoded_embedding.shape[-2]
            return pooler
        return encoded_embedding

class MMMM(nn.Module):
    def __init__(self):
        super(MMMM, self).__init__()

In [83]:
eeg_enc = EEGEncoder().to("cuda")

In [84]:
eeg_enc.add_task("EEG-TXT", TaskHead(input_dim=840, output_dim=1024).to("cuda"))
eeg_enc.add_task("EEG-IMG", TaskHead(input_dim=128, output_dim=1024).to("cuda"))

In [72]:
image_net_data = image_net_dataloader.load_data()
input_data_batched = image_net_data["data"]
input_data_batched_converted = torch.zeros(tuple([len(input_data_batched)]) + input_data_batched[0].shape).to(device)
for i in range(len(input_data_batched)):
  input_data_batched_converted[i] = input_data_batched[i].to(device)
target_batched = image_net_data["target"]
target_batched_converted = torch.zeros(tuple([len(target_batched)]) + target_batched[0].shape).to(device)
for i in range(len(target_batched)):
  target_batched_converted[i] = target_batched[i].to(device)

In [85]:
encoded_embedding = eeg_enc("EEG-IMG", input_data_batched_converted.to(device).float(), pool_result=True)